# 🔬 D2Vformer — Phase 2: Scientific Diagnostic Analysis
## Attention Entropy, Ablations & Temperature Scaling

**Prerequisites:**
1. Upload your full `D2Vformer/` folder to Google Drive (e.g. `My Drive/D2Vformer/`)
2. Set **Runtime → Change runtime type → T4 GPU**
3. Run cells top to bottom ▶

| Experiment | Script | What it measures |
|---|---|---|
| **Diagnostics 1,3,4,5** | `attention_diagnostics.py` | H_norm, N_eff, heatmaps, entropy-MSE correlation |
| **Diagnostic 6** | `uniform_attention_control.py` | Learned vs. uniform attention utility |
| **Diagnostic 7** | `shuffled_attention_control.py` | Does temporal index alignment matter? |
| **Phase 3** | Temperature ablation (inline) | tau=0.5/1/2/4/learnable |

> ⚠️ Phase-1 checkpoints (`pured2vformer_ETTh1_seed42.pt` etc.) must be inside your Drive folder at `D2Vformer/results/checkpoints/`. If missing, Cell 3 will train them automatically.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Mount Google Drive & Configure Project Root
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys

# ✏️  CHANGE THIS if your folder is somewhere else on Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/D2Vformer'

if not os.path.isdir(DRIVE_PROJECT_PATH):
    raise FileNotFoundError(
        f"Folder not found: {DRIVE_PROJECT_PATH}\n"
        "Upload your D2Vformer folder to Drive and update DRIVE_PROJECT_PATH above."
    )

PROJECT_ROOT = DRIVE_PROJECT_PATH
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root : {PROJECT_ROOT}')
print(f'Contents     : {os.listdir(PROJECT_ROOT)}')

ckpt_dir = os.path.join(PROJECT_ROOT, 'results', 'checkpoints')
if os.path.isdir(ckpt_dir):
    ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pt')]
    print(f'Checkpoints  : {len(ckpts)} found -> {ckpts}')
else:
    print('WARNING: No checkpoints folder yet. Cell 3 will train them.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: Install dependencies & quick import check
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'matplotlib', 'tqdm', 'scipy'],
    check=True
)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU. Set Runtime -> T4 GPU for speed.')

# Verify imports from the Drive folder work
from models.pure_d2vformer import PureD2Vformer
from models.temperature_d2vformer import TemperaturePureD2Vformer
from utils.data import get_data_loaders
from utils.reproducibility import set_seed, compute_parameter_checksum

m  = PureD2Vformer(c_in=7, seq_len=96)
x  = torch.randn(2, 96, 7)
xm = torch.randn(2, 96, 4)
ym = torch.randn(2, 48, 4)
out, A = m(x, xm, ym)
assert out.shape == (2, 48, 7)
assert A.shape   == (2, 128, 48, 96)
params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'PureD2Vformer import OK  (params={params:,})')
del m, x, xm, ym, out, A


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 (Optional): Train Phase-1 PureD2Vformer checkpoints if missing
# Skip if checkpoints already exist in results/checkpoints/
# ─────────────────────────────────────────────────────────────────────────────
import os, time
import torch, torch.nn as nn

from models.pure_d2vformer import PureD2Vformer
from utils.data import get_data_loaders
from utils.reproducibility import set_seed, compute_parameter_checksum

ckpt_dir = os.path.join(PROJECT_ROOT, 'results', 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)

needed = [
    f'pured2vformer_{ds}_seed{s}.pt'
    for ds in ['ETTh1', 'exchange']
    for s  in [42, 43, 44]
]
missing = [f for f in needed if not os.path.exists(os.path.join(ckpt_dir, f))]

if not missing:
    print('All 6 Phase-1 checkpoints already present — skipping training.')
else:
    print(f'{len(missing)} checkpoint(s) missing: {missing}')
    print('Starting Phase-1 training...\n')

    SEQ_LEN=96; TRAIN_HOR=48; D_MODEL=128; D_FF=256; K_FREQ=16
    DROPOUT=0.05; LR=1e-3; EPOCHS=10; PATIENCE=3; BS=64

    for dataset in ['ETTh1', 'exchange']:
        for seed in [42, 43, 44]:
            ckpt_path = os.path.join(ckpt_dir, f'pured2vformer_{dataset}_seed{seed}.pt')
            if os.path.exists(ckpt_path):
                print(f'  Skip: {os.path.basename(ckpt_path)}')
                continue
            set_seed(seed)
            train_ldr, val_ldr, _, meta = get_data_loaders(
                dataset_name=dataset, seq_len=SEQ_LEN, pred_len=TRAIN_HOR,
                batch_size=BS, data_root=os.path.join(PROJECT_ROOT, 'datasets')
            )
            model = PureD2Vformer(
                c_in=meta['num_variables'], seq_len=SEQ_LEN,
                d_model=D_MODEL, d_ff=D_FF, k_freq=K_FREQ, dropout=DROPOUT
            ).to(DEVICE)
            opt = torch.optim.Adam(model.parameters(), lr=LR)
            crit = nn.MSELoss()
            best_val, no_imp, best_ep = float('inf'), 0, 0
            t0 = time.time()
            for epoch in range(1, EPOCHS+1):
                model.train()
                for bx,by,bxm,bym in train_ldr:
                    bx,by,bxm,bym = bx.to(DEVICE),by.to(DEVICE),bxm.to(DEVICE),bym.to(DEVICE)
                    opt.zero_grad(); out,_ = model(bx,bxm,bym); crit(out,by).backward(); opt.step()
                model.eval(); vl,nv = 0.0,0
                with torch.no_grad():
                    for bx,by,bxm,bym in val_ldr:
                        bx,by,bxm,bym = bx.to(DEVICE),by.to(DEVICE),bxm.to(DEVICE),bym.to(DEVICE)
                        out,_ = model(bx,bxm,bym); vl += crit(out,by).item()*bx.size(0); nv += bx.size(0)
                vl /= max(1,nv)
                if vl < best_val:
                    best_val,no_imp,best_ep = vl,0,epoch
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'checksum': compute_parameter_checksum(model),
                        'param_count': sum(p.numel() for p in model.parameters() if p.requires_grad),
                        'c_in': meta['num_variables'], 'seq_len': SEQ_LEN,
                        'd_model': D_MODEL, 'd_ff': D_FF, 'k_freq': K_FREQ,
                        'dropout': DROPOUT, 'train_horizon': TRAIN_HOR,
                        'seed': seed, 'dataset': dataset
                    }, ckpt_path)
                else:
                    no_imp += 1
                    if no_imp >= PATIENCE: break
            print(f'  [{dataset} seed={seed}] best_val={best_val:.4f} ep={best_ep} ({time.time()-t0:.0f}s)')
    print('\nAll checkpoints ready.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4: DIAGNOSTICS 1,3,4,5 — Entropy, N_eff, Heatmaps, Entropy-MSE Correlation
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_ROOT
print('='*70)
print('RUNNING: attention_diagnostics.py')
print('='*70)
r = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'experiments', 'attention_diagnostics.py')],
    env=env, cwd=PROJECT_ROOT
)
print(f'\nReturn code: {r.returncode}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5: DIAGNOSTIC 6 — Uniform Attention Control
# Learned attention vs. perfect uniform A = 1/L  (NO retraining)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_ROOT
print('='*70)
print('RUNNING: uniform_attention_control.py')
print('='*70)
r = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'experiments', 'uniform_attention_control.py')],
    env=env, cwd=PROJECT_ROOT
)
print(f'\nReturn code: {r.returncode}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6: DIAGNOSTIC 7 — Shuffled Attention Control
# Shuffle temporal index l in learned A  (NO retraining)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_ROOT
print('='*70)
print('RUNNING: shuffled_attention_control.py')
print('='*70)
r = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'experiments', 'shuffled_attention_control.py')],
    env=env, cwd=PROJECT_ROOT
)
print(f'\nReturn code: {r.returncode}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7: Print all diagnostic CSV results
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd, os

diag_dir = os.path.join(PROJECT_ROOT, 'results', 'diagnostics')
files = {
    'Attention Entropy Metrics'  : 'attention_metrics_summary.csv',
    'Uniform Attention Control'  : 'uniform_attention_control_results.csv',
    'Shuffled Attention Control' : 'shuffled_attention_control_results.csv',
}
for title, fname in files.items():
    fp = os.path.join(diag_dir, fname)
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        print(f'\n{"="*70}\n{title}\n{"="*70}')
        print(df.to_string(index=False))
    else:
        print(f'Not found yet: {fp}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8: Display attention heatmap plots
# ─────────────────────────────────────────────────────────────────────────────
import os, glob, matplotlib.pyplot as plt, matplotlib.image as mpimg

plot_dir = os.path.join(PROJECT_ROOT, 'results', 'diagnostics', 'plots')
pngs = sorted(glob.glob(os.path.join(plot_dir, '*.png')))
if not pngs:
    print('No plots found — run Cell 4 first.')
else:
    print(f'Showing {len(pngs)} plot(s):')
    for path in pngs:
        img = mpimg.imread(path)
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(img); ax.axis('off')
        ax.set_title(os.path.basename(path), fontsize=10)
        plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9: PHASE 3 — Temperature-Scaled Attention Ablation
# Trains TemperaturePureD2Vformer at 5 tau settings, evaluates zero-shot
#   tau=0.5   sharper attention
#   tau=1.0   baseline (same as vanilla PureD2Vformer)
#   tau=2.0   softer
#   tau=4.0   very soft (approaches uniform)
#   learnable tau optimised end-to-end
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, time
import torch, numpy as np, pandas as pd
sys.path.insert(0, PROJECT_ROOT)

from experiments.temperature_experiment import (
    train_temperature_d2vformer,
    evaluate_temperature_zeroshot
)

DATASETS      = ['ETTh1', 'exchange']
SEEDS         = [42, 43, 44]
EVAL_HORIZONS = [24, 48, 96, 192, 336, 720]
TEMP_VARIANTS = [
    ('fixed',     0.5),
    ('fixed',     1.0),
    ('fixed',     2.0),
    ('fixed',     4.0),
    ('learnable', 1.0),
]
BS       = 64
CKPT_DIR = os.path.join(PROJECT_ROOT, 'results', 'checkpoints')
OUT_DIR  = os.path.join(PROJECT_ROOT, 'results', 'temperature')
os.makedirs(OUT_DIR, exist_ok=True)

records = []
for dataset in DATASETS:
    for mode, tau_init in TEMP_VARIANTS:
        label = f'tau{tau_init}' if mode == 'fixed' else 'learnable'
        print(f'\n[{dataset} | {label}] Training 3 seeds...')
        for seed in SEEDS:
            ckpt_path, train_time, param_count, final_tau = train_temperature_d2vformer(
                dataset_name=dataset, seq_len=96, train_horizon=48,
                d_model=128, d_ff=256, k_freq=16, dropout=0.05,
                temperature_mode=mode, initial_temperature=tau_init,
                lr=1e-3, epochs=10, patience=3, batch_size=BS,
                seed=seed, device=DEVICE, checkpoint_dir=CKPT_DIR
            )
            print(f'  seed={seed}: {train_time:.0f}s | final_tau={final_tau:.4f}')
            for O in EVAL_HORIZONS:
                m = evaluate_temperature_zeroshot(
                    ckpt_path=ckpt_path, eval_horizon=O,
                    dataset_name=dataset,
                    batch_size=32 if O >= 336 else BS, device=DEVICE
                )
                records.append({
                    'dataset': dataset, 'variant': label,
                    'temperature_mode': mode, 'initial_tau': tau_init,
                    'final_tau': round(m['tau'], 4), 'seed': seed,
                    'eval_horizon': O,
                    'mse': round(m['mse'], 5), 'mae': round(m['mae'], 5),
                    'H_norm': round(m['H_norm'], 5),
                    'N_eff': round(m['N_eff'], 2),
                    'N_eff_ratio': round(m['N_eff_ratio'], 4),
                    'max_attention': round(m['max_attention'], 5),
                    'checksum_verified': m['checksum_verified']
                })
                print(f'    O={O:3d}: MSE={m["mse"]:.4f} MAE={m["mae"]:.4f} H_norm={m["H_norm"]:.4f}')

df_temp = pd.DataFrame(records)
csv_out = os.path.join(OUT_DIR, 'temperature_ablation_results.csv')
df_temp.to_csv(csv_out, index=False)
print(f'\nSaved -> {csv_out}')

print('\n' + '='*70)
print('TEMPERATURE ABLATION SUMMARY (Mean across 3 Seeds)')
print('='*70)
summary = df_temp.groupby(['dataset','variant','eval_horizon'])[['mse','mae','H_norm','N_eff_ratio']].mean().round(4)
print(summary.to_string())


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10: Plot Temperature Ablation
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt, os

csv_path = os.path.join(PROJECT_ROOT, 'results', 'temperature', 'temperature_ablation_results.csv')
if not os.path.exists(csv_path):
    print('No results yet — run Cell 9.')
else:
    df = pd.read_csv(csv_path)
    mean_df = df.groupby(['dataset','variant','eval_horizon'])[['mse','H_norm']].mean().reset_index()
    COLORS  = {'tau0.5':'#e63946','tau1.0':'#457b9d','tau2.0':'#2a9d8f','tau4.0':'#e9c46a','learnable':'#6a0572'}
    MARKERS = {'tau0.5':'o','tau1.0':'s','tau2.0':'^','tau4.0':'D','learnable':'*'}
    horizons = sorted(df['eval_horizon'].unique())
    datasets = list(df['dataset'].unique())
    variants = list(df['variant'].unique())
    fig, axes = plt.subplots(len(datasets), 2, figsize=(14, 5*len(datasets)))
    if len(datasets) == 1: axes = [axes]
    for row, ds in enumerate(datasets):
        sub = mean_df[mean_df['dataset'] == ds]
        for var in variants:
            v = sub[sub['variant'] == var]
            c = COLORS.get(var,'#333'); mk = MARKERS.get(var,'o')
            axes[row][0].plot(v['eval_horizon'], v['mse'], color=c, marker=mk, label=var, lw=2)
            axes[row][1].plot(v['eval_horizon'], v['H_norm'], color=c, marker=mk, label=var, lw=2)
        for col, (ylabel, title) in enumerate([('MSE','MSE vs Horizon'),('H_norm','Entropy vs Horizon')]):
            axes[row][col].set_title(f'{ds} — {title}', fontweight='bold')
            axes[row][col].set_xlabel('Prediction Horizon O')
            axes[row][col].set_ylabel(ylabel)
            axes[row][col].legend(fontsize=9)
            axes[row][col].grid(alpha=0.3)
            axes[row][col].set_xticks(horizons)
        axes[row][1].set_ylim(0, 1.05)
    plt.suptitle('D2Vformer Phase 3: Temperature Scaling Ablation', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plot_out = os.path.join(PROJECT_ROOT, 'results', 'temperature', 'temperature_ablation_plot.png')
    plt.savefig(plot_out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {plot_out}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11: Consolidated Research Findings
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd, os

diag_dir = os.path.join(PROJECT_ROOT, 'results', 'diagnostics')
temp_dir = os.path.join(PROJECT_ROOT, 'results', 'temperature')
print('='*70 + '\nPHASE 2 CONSOLIDATED FINDINGS\n' + '='*70)

ef = os.path.join(diag_dir, 'attention_metrics_summary.csv')
if os.path.exists(ef):
    df = pd.read_csv(ef)
    print('\n[FINDING 1] Attention Entropy (mean across seeds & horizons):')
    print(df.groupby('dataset')[['H_norm','N_eff_ratio','pearson_corr']].mean().round(4).to_string())
    print('  H_norm=1.0 -> uniform | 0.0 -> fully concentrated')

uf = os.path.join(diag_dir, 'uniform_attention_control_results.csv')
if os.path.exists(uf):
    df = pd.read_csv(uf)
    print('\n[FINDING 2] Uniform Control: Mean MSE delta (uniform - learned) / learned * 100:')
    print(df.groupby('dataset')['pct_change_mse'].mean().round(2).to_string())
    print('% cases learned > uniform:')
    print(df.groupby('dataset')['learned_better'].mean().mul(100).round(1).to_string())

sf = os.path.join(diag_dir, 'shuffled_attention_control_results.csv')
if os.path.exists(sf):
    df = pd.read_csv(sf)
    print('\n[FINDING 3] Shuffled Control: Mean MSE delta:')
    print(df.groupby('dataset')['pct_change_mse'].mean().round(2).to_string())
    print('% cases alignment matters:')
    print(df.groupby('dataset')['alignment_matters'].mean().mul(100).round(1).to_string())

tf = os.path.join(temp_dir, 'temperature_ablation_results.csv')
if os.path.exists(tf):
    df = pd.read_csv(tf)
    bpd = df.groupby(['dataset','variant'])['mse'].mean().reset_index()
    print('\n[FINDING 4] Temperature Ablation — Mean MSE across all horizons:')
    for ds in bpd['dataset'].unique():
        sub = bpd[bpd['dataset']==ds].sort_values('mse')
        print(f'\n  {ds}:')
        for _,row in sub.iterrows():
            mark = ' <- BEST' if row['mse'] == sub['mse'].min() else ''
            print(f'    {row["variant"]:12s}: MSE = {row["mse"]:.4f}{mark}')

print('\n' + '='*70)
print('Results saved in: results/diagnostics/ and results/temperature/')
print('='*70)


## ✅ Results Checklist
After running all cells, your Drive folder will have:
- `results/diagnostics/attention_metrics_summary.csv`
- `results/diagnostics/uniform_attention_control_results.csv`
- `results/diagnostics/shuffled_attention_control_results.csv`
- `results/diagnostics/plots/*.png`
- `results/temperature/temperature_ablation_results.csv`
- `results/temperature/temperature_ablation_plot.png`

Paste the findings from Cell 11 back here and we'll write `RESEARCH_FINDINGS.md` together.